In [1]:
from pathlib import Path
from getpass import getpass
from typing import Literal
import json
import math
import os
import re
import time

import numpy as np
import pandas as pd
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_DIR = PROJECT_ROOT / 'data/clean/fourlang/synthetic_app_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = OUTPUT_DIR / 'synthetic_fourlang_raw.csv'
CANDIDATE_PATH = OUTPUT_DIR / 'synthetic_fourlang_candidates.csv'
AUDIT_PATH = OUTPUT_DIR / 'synthetic_fourlang_audited.csv'
GEN_MODEL = os.getenv('GEN_MODEL', 'qwen-plus').strip()
AUDIT_MODEL = os.getenv('AUDIT_MODEL', 'qwen-max').strip()
BASE_URL = os.getenv('LLM_BASE_URL', '').strip() or 'https://dashscope.aliyuncs.com/compatible-mode/v1'
API_KEY = os.getenv('LLM_API_KEY', '').strip() or os.getenv('OPENAI_API_KEY', '').strip()
TARGET_ROWS = 5000
GENERATION_BATCH_SIZE = 20
AUDIT_BATCH_SIZE = 5
MAX_RETRIES = 4
REQUEST_DELAY_SECONDS = 0.3
SEED = 42
if not API_KEY:
    API_KEY = getpass('DashScope API key: ').strip()
assert API_KEY, 'API key is required.'
client = OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=120.0, max_retries=0)
print('Generation model:', GEN_MODEL)
print('Audit model:', AUDIT_MODEL)
print('Target rows:', TARGET_ROWS)
print('Estimated generation requests:', math.ceil(TARGET_ROWS / GENERATION_BATCH_SIZE))
print('Estimated audit requests:', math.ceil(TARGET_ROWS / AUDIT_BATCH_SIZE))
print('Output:', OUTPUT_DIR)

D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generation model: qwen-plus
Audit model: qwen-max
Target rows: 5000
Estimated generation requests: 250
Estimated audit requests: 1000
Output: D:\dev\projects\fourlang_translation\data\clean\fourlang\synthetic_app_v1


In [2]:
DOMAINS = [
    'daily_chat', 'introductions', 'family_friends', 'travel', 'airport',
    'public_transport', 'taxi_driving', 'hotel', 'restaurant', 'shopping',
    'payment_banking', 'work', 'study', 'technology_app', 'customer_service',
    'delivery_logistics', 'dates_time_numbers', 'weather_plans',
    'health_basic', 'emergency_help', 'government_basic_services', 'social_events',
]

class GeneratedItem(BaseModel):
    en: str = Field(min_length=2, max_length=300)
    zh: str = Field(min_length=1, max_length=300)
    ru: str = Field(min_length=2, max_length=400)
    uz: str = Field(min_length=2, max_length=400)
    register: Literal['neutral', 'informal', 'polite']

class GeneratedBatch(BaseModel):
    items: list[GeneratedItem]

GENERATION_SYSTEM_PROMPT = '''You create original, natural, short parallel text for a real-time translation app.
For every English utterance, provide faithful Simplified Chinese, Russian, and Latin-script Uzbek translations.
Requirements:
- Create original everyday utterances, not quotations or copied text.
- English should usually contain 3-25 words. Mix statements, questions, requests, and short messages.
- Preserve every number, date, time, currency, proper noun, negation, and politeness level.
- Uzbek must be natural modern Uzbek in Latin script, never Turkish and never Cyrillic.
- Chinese must be Simplified Chinese. Russian must be natural Russian in Cyrillic.
- Avoid unsafe instructions, medical diagnosis, political persuasion, explicit sexual content, and current facts.
- Do not use placeholders such as {name}. Do not repeat or paraphrase another item in the batch.
Return JSON only: {"items":[{"en":"...","zh":"...","ru":"...","uz":"...","register":"neutral|informal|polite"}]}'''

def parse_json_object(text):
    text = (text or '').strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.IGNORECASE)
    return json.loads(text)

def call_json(model, system_prompt, payload, validator, temperature):
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': json.dumps(payload, ensure_ascii=False)},
                ],
                response_format={'type': 'json_object'},
                temperature=temperature,
            )
            return validator.model_validate(parse_json_object(response.choices[0].message.content))
        except Exception as exc:
            last_error = exc
            if getattr(exc, 'status_code', None) in {400, 401, 403, 404}:
                raise RuntimeError(f'Non-retryable API error: {exc}') from exc
            if attempt + 1 < MAX_RETRIES:
                time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f'API call failed after {MAX_RETRIES} attempts: {last_error}')

def generate_batch(domain, count, batch_number):
    payload = {
        'domain': domain,
        'count': count,
        'batch_number': batch_number,
        'variation': 'Use varied speakers, sentence structures, lengths, numbers, and politeness levels.',
    }
    return call_json(GEN_MODEL, GENERATION_SYSTEM_PROMPT, payload, GeneratedBatch, 0.8).items

test_generation = generate_batch(DOMAINS[0], 5, 0)
display(pd.DataFrame([item.model_dump() for item in test_generation]))

C:\Users\WingYouther\AppData\Local\Temp\ipykernel_11380\1550575367.py:9: UserWarning: Field name "register" in "GeneratedItem" shadows an attribute in parent "BaseModel"
  class GeneratedItem(BaseModel):


,en,zh,ru,uz,register
0,Could you please pass the salt? It's right nex...,麻烦您把盐递给我好吗？就在胡椒瓶旁边。,"Не могли бы вы, пожалуйста, передать мне соль?...","Iltimos, tuzni menga bering. U qalay qo'yilgan...",polite
1,I’ll be back in 12 minutes — just grabbing cof...,我12分钟后回来——就去楼下咖啡馆买杯咖啡。,Я вернусь через 12 минут — просто возьму кофе ...,Men 12 daqiqadan keyin qaytaman — pastdagi kaf...,neutral
2,Wait — did you say the meeting starts at 3:45 ...,等等——您说的是会议在下午3:45开始，还是4:15？,"Подождите — вы сказали, что встреча начинается...",Kutib turing — uchrashuv soat 15:45 da boshlan...,neutral
3,"No, thanks — I’ve already eaten three slices o...",不了，谢谢——我今天已经吃了三片披萨。,"Нет, спасибо — я уже съел три куска пиццы сего...","Yo'q, rahmat — men bugun allaqachon uch dona p...",informal
4,"Hi there! My name is Lena, and I’m visiting fr...",你好！我叫莱娜，从塔什干来，待三天。,"Здравствуйте! Меня зовут Лена, и я приехала из...",Salom! Mening ismim Lena va men uch kunlik Tos...,polite


In [3]:
RUN_GENERATION = True

def normalize_for_dedupe(text):
    return re.sub(r'[^\w]+', ' ', str(text).casefold(), flags=re.UNICODE).strip()

def atomic_save_csv(frame, path):
    temp = path.with_suffix('.tmp.csv')
    frame.to_csv(temp, index=False, encoding='utf-8-sig')
    temp.replace(path)

if RAW_PATH.exists():
    raw_df = pd.read_csv(RAW_PATH, keep_default_na=False)
else:
    raw_df = pd.DataFrame(columns=['pair_id', 'domain', 'register', 'en', 'zh', 'ru', 'uz', 'generation_model'])
print(f'Existing generated rows: {len(raw_df)}/{TARGET_ROWS}')
if not RUN_GENERATION:
    print('Generation is disabled. Check the test batch, then set RUN_GENERATION = True.')
else:
    seen = set(raw_df['en'].map(normalize_for_dedupe))
    max_calls = math.ceil(max(TARGET_ROWS - len(raw_df), 0) / GENERATION_BATCH_SIZE) * 2 + 10
    progress = tqdm(total=TARGET_ROWS, initial=min(len(raw_df), TARGET_ROWS), desc='Generate four-language rows')
    calls = 0
    while len(raw_df) < TARGET_ROWS and calls < max_calls:
        domain = DOMAINS[calls % len(DOMAINS)]
        count = min(GENERATION_BATCH_SIZE, TARGET_ROWS - len(raw_df))
        items = generate_batch(domain, count, calls + 1)
        additions = []
        for item in items:
            key = normalize_for_dedupe(item.en)
            if not key or key in seen:
                continue
            seen.add(key)
            additions.append({
                'pair_id': f'syn_{len(raw_df) + len(additions) + 1:06d}',
                'domain': domain,
                'register': item.register,
                'en': item.en.strip(), 'zh': item.zh.strip(),
                'ru': item.ru.strip(), 'uz': item.uz.strip(),
                'generation_model': GEN_MODEL,
            })
        if additions:
            raw_df = pd.concat([raw_df, pd.DataFrame(additions)], ignore_index=True).head(TARGET_ROWS)
            atomic_save_csv(raw_df, RAW_PATH)
            progress.update(len(additions))
        calls += 1
        time.sleep(REQUEST_DELAY_SECONDS)
    progress.close()
    print('Saved:', RAW_PATH, 'rows:', len(raw_df))

Existing generated rows: 0/5000


Generate four-language rows:   4%|▍         | 200/5000 [06:16<2:28:12,  1.85s/it]

KeyboardInterrupt: 

In [ ]:
raw_df = pd.read_csv(RAW_PATH, keep_default_na=False) if RAW_PATH.exists() else raw_df.copy()

def extract_numbers(text):
    return sorted(re.findall(r'\d+(?:[.,]\d+)?', str(text)))

han = re.compile(r'[\u4e00-\u9fff]')
cyrillic = re.compile(r'[\u0400-\u04ff]')
latin = re.compile(r'[A-Za-z]')
flags = pd.DataFrame(index=raw_df.index)
flags['empty'] = raw_df[['en', 'zh', 'ru', 'uz']].apply(lambda c: c.astype(str).str.strip().eq('')).any(axis=1)
flags['zh_script'] = ~raw_df['zh'].astype(str).map(lambda x: bool(han.search(x)))
flags['ru_script'] = ~raw_df['ru'].astype(str).map(lambda x: bool(cyrillic.search(x)))
flags['uz_script'] = raw_df['uz'].astype(str).map(lambda x: bool(cyrillic.search(x)) or not bool(latin.search(x)))
flags['number_mismatch'] = raw_df.apply(lambda r: not (extract_numbers(r.en) == extract_numbers(r.zh) == extract_numbers(r.ru) == extract_numbers(r.uz)), axis=1)
flags['duplicate_en'] = raw_df['en'].map(normalize_for_dedupe).duplicated(keep=False)
flags['too_long'] = raw_df[['en', 'zh', 'ru', 'uz']].apply(lambda c: c.astype(str).str.len().gt(400)).any(axis=1)
raw_df['automatic_flags'] = [
    '|'.join(str(name) for name, value in row.items() if bool(value))
    for _, row in flags.iterrows()
]
raw_df['automatic_pass'] = ~flags.any(axis=1)
candidate_df = raw_df[raw_df['automatic_pass']].copy().reset_index(drop=True)
atomic_save_csv(candidate_df, CANDIDATE_PATH)
display(pd.DataFrame({'count': flags.sum().astype(int), 'percent': (flags.mean() * 100).round(2)}))
print('Automatic pass:', len(candidate_df), '/', len(raw_df))
if not len(raw_df):
    print('No generated rows yet. Set RUN_GENERATION = True in the previous cell and run it first.')
display(candidate_df.head(10))

In [ ]:
class LanguageAudit(BaseModel):
    language: Literal['zh', 'ru', 'uz']
    label: Literal['correct', 'minor', 'wrong', 'junk']
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str = Field(min_length=1, max_length=220)

class RowAudit(BaseModel):
    pair_id: str
    translations: list[LanguageAudit]

class AuditBatch(BaseModel):
    items: list[RowAudit]

AUDIT_SYSTEM_PROMPT = '''You are a strict multilingual translation-data auditor.
For each English source, independently assess its Simplified Chinese, Russian, and Latin-script Uzbek translations.
correct = fully faithful and usable for machine-translation training.
minor = core meaning is preserved but wording or a non-critical detail needs review.
wrong = mistranslation, important omission/addition, entity/number/negation/politeness error, or unnatural to the point of misleading.
junk = wrong language, corrupted, or unusable.
Be especially strict that Uzbek is natural Uzbek rather than Turkish. Preserve all numbers and facts.
Return exactly one result for zh, ru, and uz per pair_id. Reasons must be concise Chinese.
Return JSON only: {"items":[{"pair_id":"...","translations":[{"language":"zh|ru|uz","label":"correct|minor|wrong|junk","confidence":0.0,"reason":"..."}]}]}'''

def audit_batch(batch_df):
    payload = {'pairs': [
        {'pair_id': r.pair_id, 'en': r.en, 'zh': r.zh, 'ru': r.ru, 'uz': r.uz}
        for r in batch_df.itertuples(index=False)
    ]}
    result = call_json(AUDIT_MODEL, AUDIT_SYSTEM_PROMPT, payload, AuditBatch, 0.0)
    expected_ids = set(batch_df['pair_id'].astype(str))
    received_ids = {item.pair_id for item in result.items}
    if received_ids != expected_ids:
        raise ValueError(f'pair_id mismatch: expected {expected_ids}, received {received_ids}')
    for item in result.items:
        languages = {x.language for x in item.translations}
        if languages != {'zh', 'ru', 'uz'}:
            raise ValueError(f'language mismatch for {item.pair_id}: {languages}')
    return result.items

test_audit_df = candidate_df.head(min(AUDIT_BATCH_SIZE, len(candidate_df)))
if len(test_audit_df):
    test_audit = audit_batch(test_audit_df)
    display(pd.DataFrame([
        {'pair_id': item.pair_id, **{f'{x.language}_{field}': getattr(x, field) for x in item.translations for field in ['label', 'confidence', 'reason']}}
        for item in test_audit
    ]))

In [ ]:
RUN_AUDIT = False
result_columns = [f'{lang}_{field}' for lang in ['zh', 'ru', 'uz'] for field in ['label', 'confidence', 'reason']] + ['audit_model', 'audit_error']
if AUDIT_PATH.exists():
    old = pd.read_csv(AUDIT_PATH, keep_default_na=False)
    old_results = old[['pair_id'] + [c for c in result_columns if c in old.columns]].drop_duplicates('pair_id', keep='last')
    audit_df = candidate_df.drop(columns=[c for c in result_columns if c in candidate_df.columns], errors='ignore').merge(old_results, on='pair_id', how='left')
else:
    audit_df = candidate_df.copy()
for col in result_columns:
    if col not in audit_df.columns:
        audit_df[col] = ''
    audit_df[col] = audit_df[col].fillna('')
completed_mask = audit_df[['zh_label', 'ru_label', 'uz_label']].isin(['correct', 'minor', 'wrong', 'junk']).all(axis=1)
print(f'Audit completed: {int(completed_mask.sum())}/{len(audit_df)}')
if not RUN_AUDIT:
    print('Audit is disabled. Check the test audit, then set RUN_AUDIT = True.')
else:
    pending = audit_df.index[~completed_mask].tolist()
    batches = [pending[i:i + AUDIT_BATCH_SIZE] for i in range(0, len(pending), AUDIT_BATCH_SIZE)]
    for indices in tqdm(batches, desc='Audit four-language rows'):
        batch_df = audit_df.loc[indices]
        try:
            results = audit_batch(batch_df)
            by_id = {item.pair_id: item for item in results}
            for idx in indices:
                item = by_id[str(audit_df.at[idx, 'pair_id'])]
                by_language = {x.language: x for x in item.translations}
                for lang in ['zh', 'ru', 'uz']:
                    result = by_language[lang]
                    audit_df.at[idx, f'{lang}_label'] = result.label
                    audit_df.at[idx, f'{lang}_confidence'] = result.confidence
                    audit_df.at[idx, f'{lang}_reason'] = result.reason
                audit_df.at[idx, 'audit_model'] = AUDIT_MODEL
                audit_df.at[idx, 'audit_error'] = ''
        except Exception as exc:
            for idx in indices:
                audit_df.at[idx, 'audit_error'] = str(exc)[:500]
        atomic_save_csv(audit_df, AUDIT_PATH)
        time.sleep(REQUEST_DELAY_SECONDS)
    print('Saved:', AUDIT_PATH)

In [ ]:
audit_df = pd.read_csv(AUDIT_PATH, keep_default_na=False) if AUDIT_PATH.exists() else audit_df.copy()
for lang in ['zh', 'ru', 'uz']:
    audit_df[f'{lang}_confidence'] = pd.to_numeric(audit_df[f'{lang}_confidence'], errors='coerce').fillna(0.0)
complete = audit_df[['zh_label', 'ru_label', 'uz_label']].isin(['correct', 'minor', 'wrong', 'junk']).all(axis=1)
high_conf_correct = pd.Series(True, index=audit_df.index)
for lang in ['zh', 'ru', 'uz']:
    high_conf_correct &= audit_df[f'{lang}_label'].eq('correct') & audit_df[f'{lang}_confidence'].ge(0.90)
accepted = audit_df[complete & high_conf_correct].copy()
rejected = audit_df[complete & audit_df[['zh_label', 'ru_label', 'uz_label']].isin(['wrong', 'junk']).any(axis=1)].copy()
needs_review = audit_df[complete & ~(high_conf_correct | audit_df.index.isin(rejected.index))].copy()
atomic_save_csv(accepted, OUTPUT_DIR / 'synthetic_fourlang_accepted.csv')
atomic_save_csv(rejected, OUTPUT_DIR / 'synthetic_fourlang_rejected.csv')
atomic_save_csv(needs_review, OUTPUT_DIR / 'synthetic_fourlang_needs_human_review.csv')
print('Complete:', int(complete.sum()))
print('Accepted:', len(accepted))
print('Rejected:', len(rejected))
print('Needs human review:', len(needs_review))
for lang in ['zh', 'ru', 'uz']:
    display(audit_df[complete][f'{lang}_label'].value_counts().rename(lang).to_frame())

directions = [('en', 'uz'), ('uz', 'en'), ('en', 'ru'), ('ru', 'en'), ('en', 'zh'), ('zh', 'en')]
directional_rows = []
for row in accepted.itertuples(index=False):
    values = {'en': row.en, 'zh': row.zh, 'ru': row.ru, 'uz': row.uz}
    for src_lang, tgt_lang in directions:
        directional_rows.append({
            'pair_id': row.pair_id, 'domain': row.domain,
            'src_lang': src_lang, 'tgt_lang': tgt_lang,
            'src_text': values[src_lang], 'tgt_text': values[tgt_lang],
            'direction': f'{src_lang}-{tgt_lang}', 'source_type': 'synthetic_qwen_audited',
        })
directional = pd.DataFrame(directional_rows)
if len(directional):
    rng = np.random.default_rng(SEED)
    validation_ids = set(rng.choice(accepted['pair_id'].unique(), size=max(1, round(len(accepted) * .05)), replace=False))
    directional['split'] = np.where(directional['pair_id'].isin(validation_ids), 'validation', 'train')
    for split in ['train', 'validation']:
        part = directional[directional['split'] == split]
        part.to_json(OUTPUT_DIR / f'{split}_directional.jsonl', orient='records', lines=True, force_ascii=False)
    display(directional.groupby(['split', 'direction']).size().rename('rows').reset_index())

manifest = {
    'created_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'generation_model': GEN_MODEL, 'audit_model': AUDIT_MODEL, 'base_url': BASE_URL,
    'target_rows': TARGET_ROWS, 'accepted_rows': len(accepted),
    'directions': [f'{a}-{b}' for a, b in directions],
    'commercial_clearance': 'Verify the API provider terms and retain generation provenance before production use.',
}
(OUTPUT_DIR / 'dataset_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('No FLORES or NTREX evaluation sentences were used for training.')